# 05 — Split-conformal calibration (CQR) + reliability
Restores coverage on test; PIT/ECE; calibrated event probabilities. Writes `conformal_metrics.json`.

In [ ]:
# === Colab/local auto-setup (device + data path) ===
import sys, os, subprocess
from pathlib import Path
def _pip(*pkgs):
    for p in pkgs:
        mod = p.split('==')[0].replace('-', '_').replace('scikit_learn', 'sklearn')
        try:
            __import__(mod)
        except Exception:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p])
_pip('numpy', 'pandas', 'scipy', 'scikit-learn', 'statsmodels', 'torch', 'matplotlib')
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
CSV = 'Bangladesh Meterological data.csv'
cands = [Path.cwd()/CSV, Path('/content')/CSV, Path(r'd:\BUET RESEARCH WORK\Bangladesh Flood')/CSV]
root = next((c.parent for c in cands if c.exists()), None)
if root is None:
    try:
        from google.colab import files
        files.upload(); root = Path.cwd()
    except Exception:
        raise FileNotFoundError('Upload "%s" next to this notebook.' % CSV)
os.environ['DFAA_ROOT'] = str(root)
print('DFAA_ROOT =', root)


In [ ]:
%%writefile common.py
"""Common config, data loader, and leak-free helpers for the Bangladesh DFAA study.

All paths hardcoded (workspace convention). Train-only fits everywhere.
Notation matches EXPERIMENT_DESIGN.md. No experiment numbers are produced here;
this is the shared, smoke-testable core that the notebooks reuse.
"""
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd

# ROOT overridable so the shipped notebooks run on Colab/local (set env DFAA_ROOT).
ROOT = Path(os.environ.get("DFAA_ROOT", r"d:\BUET RESEARCH WORK\Bangladesh Flood"))
RAW_CSV = ROOT / "Bangladesh Meterological data.csv"
ART = ROOT / "artefacts"
ART.mkdir(exist_ok=True)

# ---- locked study constants (EXPERIMENT_DESIGN.md defaults) ----
SEED = 0
VARS_Z = ["Rainfall_mm", "Soil_moisture_mm"]          # standardized by train climatology
# robust standardization: per-(s,m) sd floored at SD_FLOOR_FRAC * station-pooled train sd,
# then z clipped to +-Z_CLIP. Guards the soil-moisture saturation / dry-month near-zero-sd
# pathology (otherwise z -> ~-40000). Fixed/train-only transforms => no leakage; train cells
# (|z|<3.6) are untouched. Documented in RESULTS_LOG S1/S2.
SD_FLOOR_FRAC = 0.15
Z_CLIP = 4.0
W_WEIGHTS = (1 / 3, 1 / 3, 1 / 3)                     # w1*z_P + w2*z_SM + w3*SPEI
ALPHA_DFAA = 1.8                                       # Wu (2006) constant in Eq. 2
LEADS = (1, 2, 3)                                      # symmetric window scale = lead h
TAUS = np.array([0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95], dtype=np.float64)
THETA_PCT = 80.0                                       # theta_D = 80th pct of |DFAA| on train

# time-ordered split by ORIGIN year (the month t at which DFAA(s,t) is anchored)
TRAIN_YEARS = (2000, 2014)
VAL_YEARS = (2015, 2017)
TEST_YEARS = (2018, 2022)

# BMD station latitudes/longitudes (deg) for PET extraterrestrial radiation + maps.
# Standard BMD station coordinates; for PET only latitude matters (Ra is ~flat to +-0.2 deg).
# Provenance flagged for final verification before the .tex (CLAUDE.md Rule 3).
STATION_LATLON = {
    "Barisal":              (22.70, 90.37),
    "Bogra":                (24.85, 89.37),
    "Chittagong(Air-port)": (22.25, 91.81),
    "Comilla":              (23.43, 91.18),
    "Cox's Bazar":          (21.45, 91.97),
    "Dhaka":                (23.78, 90.38),
    "Faridpur":             (23.60, 89.85),
    "Jessore":              (23.18, 89.16),
    "Khulna":               (22.78, 89.53),
    "Mymensingh":           (24.75, 90.43),
    "Rajshahi":             (24.37, 88.70),
    "Rangpur":              (25.73, 89.23),
    "Sylhet":               (24.90, 91.88),
}


def load_clean():
    """Load raw CSV, drop the trailing all-NaN row, sort, add integer month index t.

    Returns a tidy long DataFrame with columns:
      Station_Name, Station_Code, Year, Month, Max_Temp, Min_Temp, Rainfall_mm,
      Soil_moisture_mm, SPEI_3, s (0..12 station id), t (0-based global month index),
      origin_year, split.
    Raw file is never modified.
    """
    df = pd.read_csv(RAW_CSV)
    # drop rows that are entirely NaN in the value columns (the trailing NaN row)
    val_cols = ["Max_Temp", "Min_Temp", "Rainfall_mm", "Soil_moisture_mm", "SPEI_3"]
    before = len(df)
    df = df.dropna(subset=["Station_Name", "Year", "Month"], how="any").copy()
    df = df.dropna(subset=val_cols, how="all").copy()
    dropped = before - len(df)

    df["Year"] = df["Year"].astype(int)
    df["Month"] = df["Month"].astype(int)
    df = df.sort_values(["Station_Name", "Year", "Month"]).reset_index(drop=True)

    stations = sorted(df["Station_Name"].unique())
    sid = {name: i for i, name in enumerate(stations)}
    df["s"] = df["Station_Name"].map(sid)

    # global 0-based month index over 2000-01 .. 2022-12
    df["t"] = (df["Year"] - 2000) * 12 + (df["Month"] - 1)

    def split_of(y):
        if TRAIN_YEARS[0] <= y <= TRAIN_YEARS[1]:
            return "train"
        if VAL_YEARS[0] <= y <= VAL_YEARS[1]:
            return "val"
        return "test"

    df["origin_year"] = df["Year"]
    df["split"] = df["Year"].map(split_of)
    return df, stations, sid, dropped


def to_grid(df, col):
    """Return an (S, T) float array of `col` indexed by [station s, month index t],
    NaN where missing. S=13 stations, T=276 months (2000-01..2022-12)."""
    S = df["s"].nunique()
    T = 276
    g = np.full((S, T), np.nan, dtype=np.float64)
    g[df["s"].to_numpy(), df["t"].to_numpy()] = df[col].to_numpy(dtype=float)
    return g


def train_mask_t(years=TRAIN_YEARS):
    """Boolean length-276 mask of month indices whose calendar year is in `years`."""
    t = np.arange(276)
    yr = 2000 + t // 12
    return (yr >= years[0]) & (yr <= years[1])


def fit_climatology(grid, train_t):
    """Per (station s, calendar month m) mean/std on TRAIN months only, with a robust
    std floor at SD_FLOOR_FRAC * station-pooled train sd (guards saturated/dry near-zero-sd
    cells). grid: (S,T); train_t: bool length T. Returns mu,sd as (S,12)."""
    S, T = grid.shape
    mu = np.full((S, 12), np.nan)
    sd = np.full((S, 12), np.nan)
    months = np.arange(T) % 12
    for s in range(S):
        for m in range(12):
            sel = (months == m) & train_t
            vals = grid[s, sel]
            vals = vals[~np.isnan(vals)]
            if len(vals) >= 2:
                mu[s, m] = vals.mean()
                sd[s, m] = vals.std(ddof=1)
    glob = np.nanstd(grid[:, train_t])
    for s in range(S):
        stat_sd = np.nanstd(grid[s, train_t])
        floor = SD_FLOOR_FRAC * stat_sd if np.isfinite(stat_sd) and stat_sd > 1e-6 else glob
        floor = max(floor, 1e-6)
        for m in range(12):
            if not np.isfinite(sd[s, m]) or sd[s, m] < floor:
                sd[s, m] = floor
            if not np.isfinite(mu[s, m]):
                mu[s, m] = np.nanmean(grid[s, train_t])
    return mu, sd


def standardize(grid, mu, sd):
    """z_v(s,t) = clip( (x - mu[s,m]) / sd[s,m], -Z_CLIP, +Z_CLIP ), m = t%12. NaNs propagate."""
    S, T = grid.shape
    months = np.arange(T) % 12
    z = (grid - mu[:, months]) / sd[:, months]
    return np.clip(z, -Z_CLIP, Z_CLIP)


In [ ]:
%%writefile evalkit.py
"""Shared probabilistic-forecast evaluation (reused by Steps 3-6).

All metrics take predictive quantiles Q[N, nT] (monotone non-decreasing in tau) and targets y[N].
CRPS uses the quantile estimator CRPS ~= 2*mean_tau(pinball_tau) (EXPERIMENT_DESIGN Step 7);
coarse but identical across methods, so CRPSS comparisons are fair.
"""
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

from common import TAUS


def pinball_per_tau(Q, y, taus=TAUS):
    err = y[:, None] - Q                       # [N,nT]
    pb = np.maximum(taus[None, :] * err, (taus[None, :] - 1) * err)
    return pb.mean(axis=0)                      # [nT]


def pinball(Q, y, taus=TAUS):
    return float(pinball_per_tau(Q, y, taus).mean())


def crps(Q, y, taus=TAUS):
    return float(2.0 * pinball_per_tau(Q, y, taus).mean())


def coverage(Q, y, lo_i, hi_i):
    lo, hi = Q[:, lo_i], Q[:, hi_i]
    inside = (y >= lo) & (y <= hi)
    return float(inside.mean()), float((hi - lo).mean())


def cdf_at(Q, thr, taus=TAUS):
    """Predictive CDF at threshold thr via linear interpolation across the quantile grid."""
    Q = np.asarray(Q, float)
    N, m = Q.shape
    thr_arr = np.full(N, thr) if np.isscalar(thr) else np.asarray(thr, float)
    k = np.sum(Q < thr_arr[:, None], axis=1)
    lo = np.clip(k - 1, 0, m - 1); hi = np.clip(k, 0, m - 1)
    ar = np.arange(N)
    qlo, qhi = Q[ar, lo], Q[ar, hi]
    tlo, thi = taus[lo], taus[hi]
    gap = qhi - qlo
    frac = np.where(gap > 1e-9, (thr_arr - qlo) / np.where(gap > 1e-9, gap, 1.0), 0.0)
    cdf = tlo + frac * (thi - tlo)
    cdf = np.where(k == 0, taus[0], cdf)
    cdf = np.where(k == m, taus[-1], cdf)
    return np.clip(cdf, 0.0, 1.0)


def event_metrics(Q, y, theta, taus=TAUS):
    """DTF (y>=+theta) and FTD (y<=-theta) detection from the predictive CDF."""
    out = {}
    p_dtf = 1.0 - cdf_at(Q, theta, taus)
    p_ftd = cdf_at(Q, -theta, taus)
    for name, p, ind in [("DTF", p_dtf, (y >= theta).astype(int)),
                          ("FTD", p_ftd, (y <= -theta).astype(int))]:
        d = {"base_rate": float(ind.mean()), "brier": float(brier_score_loss(ind, np.clip(p, 0, 1)))}
        if ind.sum() > 0 and ind.sum() < len(ind):
            d["auc"] = float(roc_auc_score(ind, p))
            d["pr_auc"] = float(average_precision_score(ind, p))
        else:
            d["auc"] = float("nan"); d["pr_auc"] = float("nan")
        out[name] = d
    return out


def all_metrics(Q, y, theta, crps_ref=None, taus=TAUS):
    """Bundle of probabilistic + calibration + event metrics for one method/lead/split."""
    pin = pinball(Q, y, taus)
    cr = crps(Q, y, taus)
    p80, w80 = coverage(Q, y, 1, 5)   # taus index 1=0.10, 5=0.90 -> 80% PI
    p90, w90 = coverage(Q, y, 0, 6)   # taus index 0=0.05, 6=0.95 -> 90% PI
    m = {"n": int(len(y)), "pinball": pin, "crps": cr,
         "picp80": p80, "width80": w80, "picp90": p90, "width90": w90}
    if crps_ref is not None and crps_ref > 0:
        m["crpss"] = float(1.0 - cr / crps_ref)
    m["event"] = event_metrics(Q, y, theta, taus)
    return m


def residual_quantiles(resid_train, taus=TAUS):
    """Empirical quantiles of training residuals -> additive spread for a point forecast."""
    r = resid_train[np.isfinite(resid_train)]
    return np.quantile(r, taus)


def point_to_quantiles(point, resid_q):
    """Q[N,nT] = point[:,None] + resid_q[None,:] (homoscedastic residual probabilization)."""
    Q = point[:, None] + resid_q[None, :]
    return np.maximum.accumulate(Q, axis=1)   # enforce monotone (sorted resid_q already monotone)


In [ ]:
"""Step 5 - split-conformal quantile regression (CQR, Eq. 7-8) + reliability/PIT + event probs.

Calibration set C = the VAL block (2015-2017), disjoint from train (model fit) and test (eval).
For each symmetric quantile pair we compute a conformal radius E-hat on C and widen the test
quantiles, giving distribution-free marginal coverage (exchangeability approx; the 2018-22 shift
is itself reported). Rebuilds a calibrated CDF -> recomputes event Brier/AUC + PIT/ECE.

Writes artefacts/conformal_metrics.json + appends RESULTS_LOG.
"""
import json
import numpy as np
from common import ART, LEADS, TAUS, SEED
import evalkit as ek

np.random.seed(SEED)
print("=" * 70); print("STEP 5 - conformal calibration (CQR) + reliability"); print("=" * 70)

pp = np.load(ART / "model_preds.npz", allow_pickle=True)
meta = json.loads((ART / "dfaa_meta.json").read_text())
THETA = {h: meta["theta_D"][str(h)] for h in LEADS}

# symmetric (lo_idx, hi_idx, alpha) for the 7-tau grid {.05,.10,.25,.50,.75,.90,.95}
PAIRS = [(0, 6, 0.10), (1, 5, 0.20), (2, 4, 0.50)]   # 90%, 80%, 50% central intervals


def cqr_radius(Qcal, ycal, lo_i, hi_i, alpha):
    E = np.maximum(Qcal[:, lo_i] - ycal, ycal - Qcal[:, hi_i])
    n = len(E)
    k = min(int(np.ceil((n + 1) * (1 - alpha))), n)
    return float(np.sort(E)[k - 1])


def apply_cqr(Qcal, ycal, Qtest):
    """Return calibrated test quantiles (widen each symmetric pair by its conformal radius)."""
    Qc = Qtest.copy()
    radii = {}
    for lo_i, hi_i, a in PAIRS:
        e = cqr_radius(Qcal, ycal, lo_i, hi_i, a)
        radii[(lo_i, hi_i)] = e
        Qc[:, lo_i] = Qtest[:, lo_i] - e
        Qc[:, hi_i] = Qtest[:, hi_i] + e
    Qc = np.sort(Qc, axis=1)   # keep monotone
    return Qc, radii


def pit_ece(Q, y, grid=np.linspace(0.05, 0.95, 19)):
    pit = ek.cdf_at(Q, y)                       # PIT values, should be ~U(0,1)
    emp = np.array([(pit <= p).mean() for p in grid])
    return float(np.mean(np.abs(emp - grid))), pit


results = {}
for model in ["gbq", "lstm"]:
    results[model] = {}
    for h in LEADS:
        Qcal = pp[f"{model}_{h}_val_Q"]; ycal = pp[f"{model}_{h}_val_y"]
        Qte = pp[f"{model}_{h}_test_Q"]; yte = pp[f"{model}_{h}_test_y"]
        # before
        p80b, w80b = ek.coverage(Qte, yte, 1, 5)
        p90b, w90b = ek.coverage(Qte, yte, 0, 6)
        ece_b, _ = pit_ece(Qte, yte)
        evb = ek.event_metrics(Qte, yte, THETA[h])
        # after CQR
        Qc, radii = apply_cqr(Qcal, ycal, Qte)
        p80a, w80a = ek.coverage(Qc, yte, 1, 5)
        p90a, w90a = ek.coverage(Qc, yte, 0, 6)
        ece_a, _ = pit_ece(Qc, yte)
        eva = ek.event_metrics(Qc, yte, THETA[h])
        crps_a = ek.crps(Qc, yte)
        results[model][h] = {
            "before": {"picp80": p80b, "width80": w80b, "picp90": p90b, "width90": w90b,
                       "ece": ece_b, "event": evb},
            "after": {"picp80": p80a, "width80": w80a, "picp90": p90a, "width90": w90a,
                      "ece": ece_a, "event": eva, "crps": crps_a},
            "cqr_radius90": radii[(0, 6)], "cqr_radius80": radii[(1, 5)],
        }

(ART / "conformal_metrics.json").write_text(json.dumps(results, indent=2, default=float))

print("\nPICP (test) BEFORE -> AFTER CQR  [target 0.80 / 0.90], width, ECE:")
for model in ["gbq", "lstm"]:
    print(f"\n=== {model.upper()} ===")
    print(f"{'h':>2} {'PICP80_b':>9} {'PICP80_a':>9} {'PICP90_b':>9} {'PICP90_a':>9} "
          f"{'w90_b':>7} {'w90_a':>7} {'ECE_b':>7} {'ECE_a':>7}")
    for h in LEADS:
        b = results[model][h]["before"]; a = results[model][h]["after"]
        print(f"{h:>2} {b['picp80']:>9.3f} {a['picp80']:>9.3f} {b['picp90']:>9.3f} {a['picp90']:>9.3f} "
              f"{b['width90']:>7.3f} {a['width90']:>7.3f} {b['ece']:>7.3f} {a['ece']:>7.3f}")

print("\nEvent detection (test) AFTER CQR - DTF/FTD Brier & AUC (AUC rank-invariant to widening):")
for model in ["gbq", "lstm"]:
    print(f"\n=== {model.upper()} ===  {'h':>1} {'DTF-Brier':>10} {'DTF-AUC':>8} {'FTD-Brier':>10} {'FTD-AUC':>8}")
    for h in LEADS:
        ev = results[model][h]["after"]["event"]
        print(f"{'':>20} {h:>1} {ev['DTF']['brier']:>10.4f} {ev['DTF']['auc']:>8.3f} "
              f"{ev['FTD']['brier']:>10.4f} {ev['FTD']['auc']:>8.3f}")
print(f"\nWrote {ART/'conformal_metrics.json'}")
print("\nSTEP 5 OK.")
